# Full-Triple-Matcher (FTM) — Version 1: Baseline Algorithm

## Academic Project Evaluation — Stage 1: Baseline Reference
**Base Paper:** *"Full Triple Matcher: Integrating All Triple Elements between Heterogeneous Knowledge Graphs"* (Yamamoto & Takeda, ACM Transactions on the Web, 2025)

---

### Key Workflow of this Notebook:
1. **Load Datasets:** Movie KG1 & KG2, predicate functionality & inversability tables.
2. **Execute Baseline Matcher:** Apply static global threshold ($\\theta_{base} = 0.80$).
3. **Evaluate Performance:** Compute Precision, Recall, F1-Score, and Hit@k on Movies.
4. **Inspect Large-Scale Benchmark:** Review precomputed Baseline metrics on the 666,322-pair DW-NB dataset.
5. **Identify Bottlenecks:** Explain why fixed thresholding creates False Negatives (low recall).

In [1]:
# 1. Setup Environment and Import Modules
import os
import sys
import json
import pandas as pd
import numpy as np

# Locate project root dynamically
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
curr = os.path.abspath(notebook_dir)
while curr and os.path.dirname(curr) != curr:
    if os.path.isdir(os.path.join(curr, "codes")) and os.path.isdir(os.path.join(curr, "dataset")):
        break
    curr = os.path.dirname(curr)
project_root = curr

entity_matching_dir = os.path.join(project_root, "codes", "entity-matching")
utils_dir = os.path.join(project_root, "codes", "utils")

for p in [project_root, entity_matching_dir, utils_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

from file_utils import load_csv, load_json, load_rdf_graph
import adaptive_threshold as at
import conflict_resolution as cr
from entity_matching import run_pipeline
from evaluation import evaluate_predictions, build_comparison_dataframe

print("Project environment loaded successfully.")
print(f"Project root: {project_root}")

Project environment loaded successfully.
Project root: d:\FTM\Full-Triple-Matcher-2


## 2. Predicate Inversability & Functionality

In FTM, predicate alignment and relational importance are determined by **inversability**:
$$\\text{inversability}(p) = \\text{functionality}(p) \\times \\text{inverse\\_functionality}(p)$$

In [2]:
# 2. Inspect Inversability Tables for Movie KGs
df_inv1 = load_csv(os.path.join(project_root, "dataset", "movies", "kg1_inv.csv"))
df_inv2 = load_csv(os.path.join(project_root, "dataset", "movies", "kg2_inv.csv"))

print("--- KG1 Predicates ---")
display(df_inv1)

print("\n--- KG2 Predicates ---")
display(df_inv2)

--- KG1 Predicates ---

--- KG2 Predicates ---


predicate,functionality,inverse_functionality,inversability
http://kg1.example.org/ontology/title,1.0,1.0,1.0
http://www.w3.org/2000/01/rdf-schema#label,1.0,1.0,1.0
http://kg1.example.org/ontology/releaseYear,1.0,0.9,0.9
http://kg1.example.org/ontology/director,1.0,0.6,0.6
http://kg1.example.org/ontology/country,1.0,0.2,0.2
http://kg1.example.org/ontology/name,1.0,1.0,1.0
http://kg1.example.org/ontology/genre,1.0,0.5,0.5
http://kg1.example.org/ontology/runtime,1.0,1.0,1.0


## 3. Load Candidate Pairs & Ground Truth Alignments

In [3]:
# 3. Load Candidates & Gold Standards
candidates_df = load_csv(os.path.join(project_root, "dataset", "movies", "movie_candidates.csv"))
gold_df = load_csv(os.path.join(project_root, "dataset", "movies", "small_movie_gold.csv"))

print(f"Loaded {len(candidates_df)} candidate pairs.")
print(f"Loaded {len(gold_df)} gold standard alignments.")
display(candidates_df.head(8))

Loaded 426 candidate pairs.
Loaded 22 gold standard alignments.


e1,e2,sim,triple_sim
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Action_Film,1.000000,1.0
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Avatar_2009,0.266667,0.0
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Bong_Joon_Ho,0.342000,0.0
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Chris_Nolan,0.288000,0.0
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Christopher_Nolan,0.288000,0.0
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Dark_Knight_2008,0.205714,0.0
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Gladiator_2000,0.320000,0.0
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Inception_2010,0.576000,0.0


## 4. Run Baseline Matching (Fixed Global Threshold $\\theta = 0.80$)

Candidates with $\\text{sim} \\ge 0.80$ are accepted; greedy tie-breaking enforces 1-to-1 matching.

In [4]:
# 4. Apply Fixed Threshold (theta = 0.80)
baseline_decisions = at.apply_adaptive_threshold(
    candidates_df=candidates_df,
    use_adaptive=False,
    base_threshold=0.80,
    entity_col1='e1',
    entity_col2='e2',
    sim_col='sim'
)

accepted = baseline_decisions[baseline_decisions['decision'] == True]
print(f"Accepted Matches (sim >= 0.80): {len(accepted)}")
display(baseline_decisions.head(8))

Accepted Matches (sim >= 0.80): 24


entity_kg1,entity_kg2,similarity,best_candidate_score,second_candidate_score,candidate_margin,base_threshold,uncertainty_penalty,confidence_bonus,adaptive_threshold,decision,triple_similarity
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Action_Film,1.000000,1.0,0.818182,0.181818,0.8,0.0,0.0,0.8,True,1.000000
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/SciFi,0.818182,1.0,0.818182,0.000000,0.8,0.0,0.0,0.8,True,0.818182
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Science_Fiction,0.818182,1.0,0.818182,0.000000,0.8,0.0,0.0,0.8,True,0.818182
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Inception_2010,0.576000,1.0,0.818182,0.000000,0.8,0.0,0.0,0.8,False,0.000000
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Romantic_Drama,0.380000,1.0,0.818182,0.000000,0.8,0.0,0.0,0.8,False,0.380000
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/James_Cameron,0.360000,1.0,0.818182,0.000000,0.8,0.0,0.0,0.8,False,0.000000
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Bong_Joon_Ho,0.342000,1.0,0.818182,0.000000,0.8,0.0,0.0,0.8,False,0.000000
http://kg1.example.org/resource/Action,http://kg2.example.org/resource/Gladiator_2000,0.320000,1.0,0.818182,0.000000,0.8,0.0,0.0,0.8,False,0.000000


## 5. Baseline Evaluation Metrics on Movie KG

In [5]:
# 5. Compute Evaluation Metrics
metrics_movies = evaluate_predictions(
    predictions_df=baseline_decisions,
    gold_df=gold_df,
    pred_col1='entity_kg1',
    pred_col2='entity_kg2',
    decision_col='decision',
    sim_col='similarity'
)

print("=== MOVIE KG: BASELINE RESULTS ===")
for k, v in metrics_movies.items():
    print(f"  {k:<20}: {v}")

=== MOVIE KG: BASELINE RESULTS ===
  total_gold          : 22
  predicted_matches   : 18
  true_positives      : 18
  false_positives     : 0
  false_negatives     : 4
  precision           : 1.0
  recall              : 0.818182
  f1                  : 0.9
  hit_at_1            : 0.9
  hit_at_10           : 1.0


## 6. Large-Scale Benchmark: Baseline on DW-NB Dataset (666k Pairs)

Without needing to re-run hours of heavy similarity compute, we review the precomputed baseline benchmark on DBpedia-to-Wikidata (25,000 gold alignments):

In [6]:
# 6. Load Precomputed DW-NB Baseline Metrics
dw_base = load_json(os.path.join(project_root, "results", "full-triple-matcher", "dw-nb", "baseline", "metrics.json"))
df_dw_base = pd.DataFrame(list(dw_base.items()), columns=["Metric", "Baseline Value"])
print("=== DW-NB BENCHMARK: BASELINE PERFORMANCE ===")
display(df_dw_base)

=== DW-NB BENCHMARK: BASELINE PERFORMANCE ===


Metric,Baseline Value
total_gold,25000
predicted_matches,28569
true_positives,13550
false_positives,15019
false_negatives,11450
precision,0.47429
recall,0.542
f1,0.50589
hit_at_1,0.9254
hit_at_10,0.96116


## 7. Baseline Bottleneck Summary

| Bottleneck | Description | Consequence |
|---|---|---|
| **Rigid Thresholding** | Candidates with similarity between 0.70 and 0.79 are rejected even with zero competition. | **Low Recall (54.20% on DW-NB)** |
| **No Dynamic Margin** | High-confidence clear winners are treated the same as close ambiguous competitors. | Missed valid matches |
| **Greedy Tie-Break** | Discards rival candidates blindly based on string order. | Multi-mapping vulnerabilities |

These issues are resolved in **Notebook 2 (Improvement 2: Adaptive Threshold Optimization)**.

In [ ]:
# Save Baseline outputs to results directory
out_dir = os.path.join(project_root, "results", "full-triple-matcher", "movies", "baseline")
os.makedirs(out_dir, exist_ok=True)

baseline_decisions.to_csv(os.path.join(out_dir, "matching_decisions.csv"), index=False)
with open(os.path.join(out_dir, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics_movies, f, indent=4)
pd.DataFrame(list(metrics_movies.items()), columns=["metric", "value"]).to_csv(os.path.join(out_dir, "summary.csv"), index=False)

print(f"[SAVED] Baseline outputs saved successfully to:")
print(f"  -> {os.path.relpath(os.path.join(out_dir, 'matching_decisions.csv'), project_root)}")
print(f"  -> {os.path.relpath(os.path.join(out_dir, 'metrics.json'), project_root)}")
print(f"  -> {os.path.relpath(os.path.join(out_dir, 'summary.csv'), project_root)}")